# Correcting P-vals for MWAS findings for *totiviridiae* family
### Author: Oscar Heath
### Date: Sept 12 2026

The file results.parquet contains 80K MWAS results for *totiviridiae*, with associated p-values. These raw p-values have not undergone multiple test correction. We use the Benjamin-Hochberg multiple test correction method to compute corrected p-values, with $\alpha = 0.05$, and save all significant findings.

In [11]:
import polars as pl
from statsmodels.stats.multitest import multipletests

In [13]:
df = pl.read_parquet("../data/results.parquet")
significant_results = df.filter(pl.col("pvalue") < 0.05)

In [14]:
pvals = df.select(pl.col("pvalue")).to_numpy().flatten()
rejected, corrected_pvals, _, _ = multipletests(pvals, alpha=0.05, method='fdr_bh')

In [15]:
# add a column to the dataframe with the corrected p-values
df = df.with_columns(pl.Series("corrected_pvalue", corrected_pvals))

# get the significant results
significant_results = df.filter(pl.col("corrected_pvalue") < 0.05)

In [16]:
significant_results.write_parquet("../data/significant_results.parquet")

In [ ]:
# rank order findings by a combination of p-value, effect size, and number of studies

# Cohen's d: pooled-variance standardized mean difference between positive/negative groups
pooled_std = (
    (
        (pl.col("count_positive") - 1) * pl.col("std_positive") ** 2
        + (pl.col("count_negative") - 1) * pl.col("std_negative") ** 2
    )
    / (pl.col("count_positive") + pl.col("count_negative") - 2)
).sqrt()

significant_results = significant_results.with_columns(
    ((pl.col("mean_positive") - pl.col("mean_negative")) / pooled_std).alias("effect_size"),
    # proxy for "number of studies" - total samples backing the finding.
    # once findings are linked across bioprojects (see link_findings.ipynb), replace this
    # with a count of independent studies supporting the same finding.
    (pl.col("count_positive") + pl.col("count_negative")).alias("num_samples"),
)

# p-value, |effect size|, and sample count live on very different scales, so rank each
# metric (0 = least important) and average the ranks into one composite score instead of
# picking arbitrary weights. descending=True on corrected_pvalue flips the direction so
# that, for all three ranks, a higher rank means more important.
n = significant_results.height
significant_results = significant_results.with_columns(
    pl.col("corrected_pvalue").rank(method="average", descending=True).alias("pvalue_rank"),
    pl.col("effect_size").abs().rank(method="average").alias("effect_size_rank"),
    pl.col("num_samples").rank(method="average").alias("num_samples_rank"),
)

significant_results = significant_results.with_columns(
    (
        (pl.col("pvalue_rank") + pl.col("effect_size_rank") + pl.col("num_samples_rank"))
        / (3 * n)
    ).alias("importance_score")
).sort("importance_score", descending=True)

significant_results

In [ ]:
# saved separately from significant_results.parquet so the embed/link pipeline
# (which reads that file fresh) is unaffected by the extra columns and new row order
significant_results.write_parquet("../data/ranked_significant_results.parquet")